In [ ]:
# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from plotly.express import line

I will use VAR method for forecasting time series. In future LSTM usage is possible.

General ideas. 
Me can observe $$a_k, b_k, p_k => a(t_k), b(t_k)$$.
Moreover we can calculate two more statistical series. Where one of them is one dimensional, 
but other is M-dimensional.

# 1. Load up data

In [ ]:
# dataset = pd.read_csv("../../src/datasets/weather/DailyDelhiClimateTest.csv")
dataset = pd.read_csv("../../src/datasets/weatherHistory.csv")

# Small enhancements
dataset.dropna(inplace=True)
try:
    dataset.index = pd.DatetimeIndex(dataset["Formatted Date"], tz="CET", name="date")
    dataset.sort_index(inplace=True)
    dataset.drop(
        columns=[
            "Formatted Date",
            "Summary",
            "Precip Type",
            "Daily Summary",
            "Loud Cover",
        ],
        inplace=True,
    )
except KeyError:
    print("Given columns are already dropped")
dataset = dataset[dataset.index < pd.to_datetime("2009-01-01 00:00:00+01:00")]
N = dataset.shape[0]
display(dataset.head())
# Get increments
df = dataset.diff()
# df.dropna(inplace=True)
df.head()

print(df.shape)

# 2. Extending dataset with addition time series (feature space replenishment)

### 2.1. New approach

In [ ]:
from magfield.forecast.algorithms import EDF

TIME = dataset.index.values
WINDOW = (24 * 7 * 4 * 4, 1)
dBx = dataset.diff()["Humidity"].dropna().values

edf = EDF(WINDOW, M=10, ord_quant=10)
edf.fit(dBx)

In [ ]:
l1 = len(edf.quants)
l2 = len(edf.probs)
l3 = len(dBx[WINDOW[0] - 1 :])
assert l1 == l3 and l1 == l2, f"Created series have different lengths: {l1}, {l2}, {l3}"
print(f"Window size: {WINDOW[0]}\nTime series length: {l1}")
print(f"Quantiles probabilities: {edf.quant_probs}")
line(edf.quants, title="Quantiles for corresponding probability levels").show()
line(edf.probs, title="Probabilities of quantiles above").show()

### 2.2. Old series

First of all we need to calculate new series. Therefore run cells below.

In [ ]:
from magfield.em.mixture import DynamicMixture
from tensorflow_probability import distributions

mix_dBX = DynamicMixture(
    num_comps=3,
    distrib=distributions.Normal,
    time_span=TIME,
    window_shape=WINDOW,
)

mix_dBX.predict_light(
    data=dBx,
    EM_params=dict(
        iter_initial=20,
        num_candid=30,
        num_best_candid=8,
        accur_final=0.05,
        prog_bar=True,
    ),
)

In [ ]:
a, b = mix_dBX.reconstruct_process_coef()
assert len(a) == len(dataset["Humidity"]) and len(a) == len(
    mix_dBX.parameters["probs"]
), "Created series have different lengths"

### 2.3. Create replenished dataframe

In [ ]:
# Get rid of unaccounted (useless) counts
if len(edf.probs) < dataset.shape[0]:
    dataset = dataset.iloc[WINDOW[0] :]

# Column names for DataFrame
col_names = {
    "edf": [f"t_{i}" for i in range(edf.M - 1)],
    "quants": [f"q_{i}" for i in range(len(edf.quant_probs))],
}
try:
    col_names["mus"] = ([f"a_{i}" for i in range(mix_dBX.num_comps)],)
    col_names["sigmas"] = ([f"b_{i}" for i in range(mix_dBX.num_comps)],)
    col_names["probs"] = ([f"p_{i}" for i in range(mix_dBX.num_comps)],)
    df_params_a = pd.DataFrame(
        mix_dBX.parameters["mus"], columns=col_names["mus"], index=dataset.index
    )
    df_params_b = pd.DataFrame(
        mix_dBX.parameters["sigmas"], columns=col_names["sigmas"], index=dataset.index
    )
    df_params_p = pd.DataFrame(
        mix_dBX.parameters["probs"], columns=col_names["probs"], index=dataset.index
    )
except BaseException:
    pass
df_density_func = pd.DataFrame(edf.probs, columns=col_names["edf"], index=dataset.index)
df_quants = pd.DataFrame(edf.quants, columns=col_names["quants"], index=dataset.index)
df_windows = pd.DataFrame(
    dBx[WINDOW[0] - 1 :], columns=["diff_Hum"], index=dataset.index
)

In [ ]:
dataset.diff().dropna()

In [ ]:
extended_Bx_dataset = pd.concat(
    [
        dataset["Humidity"],
        # df_params_a,
        # df_params_b,
        # df_params_p,
        # pd.DataFrame(a, columns=["a(t)"], index=dataset.index),
        # pd.DataFrame(b, columns=["b(t)"], index=dataset.index),
        df_density_func,
        df_quants,
        df_windows,
    ],
    axis=1,
)
display(extended_Bx_dataset.head())

Saving data to files

In [ ]:
extended_Bx_dataset.to_json("small_humidity_2006_2008.json", orient="columns")
# extended_Bx_dataset